# Extracción de Leyes (Assays) y Fusión con Intensidades Clusterizadas

Este notebook realiza la extracción de las **leyes químicas (assays)** correspondientes a la **Muestra 24 (Concentrado final de cobre)** y las fusiona con el dataset de intensidades depurado, ortogonalizado y clusterizado en el notebook anterior.

## Propósito de la Fusión:
Al cruzar las intensidades de XRF (variables crudas y ortogonales) con las leyes reales de laboratorio (ej: `%Cu` (`pCu`), `%Fe` (`pFe`), `%Zn` (`pZn`), `%Mo` (`pMo`)), se obtiene el dataset completo para:
1. Validar si los clústeres químicos/operacionales de GMM se corresponden con cambios reales en las leyes metalúrgicas.
2. Entrenar modelos de regresión predictivos (estimadores de ley a partir de XRF).

In [ ]:
import sys
import os
# Añadir el directorio raíz al path para poder importar src
sys.path.append(os.path.abspath('..'))

import pandas as pd
from src import Extractor

## 1. Conexión y Extracción de Leyes desde la Base de Datos

In [ ]:
TABLE_NAME = "works4cdp_assay"
SAMPLE_ID = 24

# Configuración de conexión idéntica a los notebooks de extracción
db_config = {
    'user': 'myuser', 
    'password': 'mypassword', 
    'host': 'localhost', 
    'port': '5432', 
    'dbname': 'mydb'
}

# Inicializar extractor y obtener assays
extractor = Extractor(table_name=TABLE_NAME, **db_config)
df_assays = extractor.get_assays(SAMPLE_ID)

print(f"Registros de leyes (assays) extraídos de la BD: {df_assays.shape[0]}")
print("Tipos de datos iniciales en df_assays:")
print(df_assays[['date', 'time']].dtypes)
df_assays.head()

Registros de leyes (assays) extraídos de la BD: 506
Tipos de datos iniciales en df_assays:
date    object
time    object
dtype: object


,date,time,n1fe,n2cu,n3zn,n4mo,n5ech5,n6sc,n7ech7,pFe,pCu,pZn,pMo,pIns,pSol
0,2021-07-05,12:02:23,46537,47247,190,2408,None,1266,None,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-07-07,11:35:33,49554,47481,240,2435,None,1233,None,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-07-12,11:25:17,42725,40545,191,3068,None,1279,None,NaN,NaN,NaN,NaN,NaN,NaN
3,2021-07-14,11:17:51,45294,45594,171,2775,None,1273,None,NaN,NaN,NaN,NaN,NaN,NaN
4,2021-07-16,11:02:24,42266,44245,245,4316,None,1289,None,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Carga de Intensidades Clusterizadas

In [ ]:
# Cargar el dataset clusterizado del notebook 03
cluster_path = 'data/processed/intensidad_cobre_24_clusterizado.csv'
df_cluster = pd.read_csv(cluster_path)

print(f"Registros del dataset clusterizado: {df_cluster.shape[0]}")
print("Tipos de datos iniciales en df_cluster:")
print(df_cluster[['date', 'time']].dtypes)
df_cluster.head()

Registros del dataset clusterizado: 495
Tipos de datos iniciales en df_cluster:
date    str
time    str
dtype: object


,date,time,instance,n5ech5,n6sc,n7ech7,n1fe_ortho,n2cu_ortho,n3zn_ortho,n4mo_ortho,cluster_kmeans,cluster_gmm
0,2021-07-05,12:02:23,1625503813,NaN,1266,NaN,5445.466427,6717.331027,-15.830887,-375.124660,0,1
1,2021-07-07,11:35:33,1625675596,NaN,1233,NaN,8122.785462,6820.772105,33.402491,-367.571895,0,1
2,2021-07-12,11:25:17,1626106980,NaN,1279,NaN,1767.280141,66.763329,-14.528884,292.536372,0,1
3,2021-07-14,11:17:51,1626279335,NaN,1273,NaN,4274.519965,5092.025343,-34.668270,-3.999489,0,1
4,2021-07-16,11:02:24,1626450270,NaN,1289,NaN,1411.213767,3806.326639,39.703426,1546.429473,2,1


## 3. Homologación de Formato de Datos y Fusión (Merge)

### ¿Por qué fallaba el merge?
SQLAlchemy devuelve los campos `date` y `time` como objetos de tipo `datetime.date` y `datetime.time`. En cambio, `pd.read_csv` los lee como strings (`object`). Al intentar fusionar columnas con tipos de datos diferentes, el resultado es vacío. 

Para solucionarlo, convertimos y normalizamos ambas columnas a string.

In [ ]:
# 1. Convertir y limpiar date/time en df_assays
df_assays['date'] = df_assays['date'].astype(str).str.strip()
df_assays['time'] = df_assays['time'].astype(str).str.strip().apply(lambda x: x.split('.')[0] if '.' in x else x)

# 2. Seleccionar columnas procesadas del clustering
cols_cluster = [
    'date', 'time', 'instance', 
    'n1fe_ortho', 'n2cu_ortho', 'n3zn_ortho', 'n4mo_ortho', 
    'cluster_kmeans', 'cluster_gmm'
]
df_cluster_subset = df_cluster[cols_cluster].copy()

# 3. Convertir y limpiar date/time en df_cluster_subset
df_cluster_subset['date'] = df_cluster_subset['date'].astype(str).str.strip()
df_cluster_subset['time'] = df_cluster_subset['time'].astype(str).str.strip().apply(lambda x: x.split('.')[0] if '.' in x else x)

print("Tipos de datos homologados:")
print("df_assays:", type(df_assays['date'].iloc[0]), type(df_assays['time'].iloc[0]))
print("df_cluster:", type(df_cluster_subset['date'].iloc[0]), type(df_cluster_subset['time'].iloc[0]))

Tipos de datos homologados:
df_assays: <class 'str'> <class 'str'>
df_cluster: <class 'str'> <class 'str'>


In [ ]:
# 4. Fusionar mediante inner join
df_merged = pd.merge(df_assays, df_cluster_subset, on=['date', 'time'], how='inner')

print(f"Dataset final fusionado. Filas: {df_merged.shape[0]}, Columnas: {df_merged.shape[1]}")
df_merged.head()

Dataset final fusionado. Filas: 495, Columnas: 22


,date,time,n1fe,n2cu,n3zn,n4mo,n5ech5,n6sc,n7ech7,pFe,...,pMo,pIns,pSol,instance,n1fe_ortho,n2cu_ortho,n3zn_ortho,n4mo_ortho,cluster_kmeans,cluster_gmm
0,2021-07-05,12:02:23,46537,47247,190,2408,None,1266,None,NaN,...,NaN,NaN,NaN,1625503813,5445.466427,6717.331027,-15.830887,-375.124660,0,1
1,2021-07-07,11:35:33,49554,47481,240,2435,None,1233,None,NaN,...,NaN,NaN,NaN,1625675596,8122.785462,6820.772105,33.402491,-367.571895,0,1
2,2021-07-12,11:25:17,42725,40545,191,3068,None,1279,None,NaN,...,NaN,NaN,NaN,1626106980,1767.280141,66.763329,-14.528884,292.536372,0,1
3,2021-07-14,11:17:51,45294,45594,171,2775,None,1273,None,NaN,...,NaN,NaN,NaN,1626279335,4274.519965,5092.025343,-34.668270,-3.999489,0,1
4,2021-07-16,11:02:24,42266,44245,245,4316,None,1289,None,NaN,...,NaN,NaN,NaN,1626450270,1411.213767,3806.326639,39.703426,1546.429473,2,1


## 4. Guardar Dataset Completo

In [ ]:
# Guardar el CSV fusionado en la carpeta de processed
output_path = 'data/processed/intensidad_cobre_24_completo.csv'
df_merged.to_csv(output_path, index=False)

print(f"Dataset completo de leyes e intensidades guardado en: {output_path}")
print("Columnas en el archivo final:")
print(list(df_merged.columns))

Dataset completo de leyes e intensidades guardado en: data/processed/intensidad_cobre_24_completo.csv
Columnas en el archivo final:
['date', 'time', 'n1fe', 'n2cu', 'n3zn', 'n4mo', 'n5ech5', 'n6sc', 'n7ech7', 'pFe', 'pCu', 'pZn', 'pMo', 'pIns', 'pSol', 'instance', 'n1fe_ortho', 'n2cu_ortho', 'n3zn_ortho', 'n4mo_ortho', 'cluster_kmeans', 'cluster_gmm']


### Siguientes pasos recomendados para la Tesis:
1. **Validación Química de los Clústeres:** Evaluar las medias de `%Cu` (`pCu`) y `%Fe` (`pFe`) por cada `cluster_gmm` mediante gráficos de cajas (*boxplots*) o pruebas de ANOVA para demostrar la significancia estadística de los grupos.
2. **Modelos Predictivos:** Ajustar modelos de regresión lineal o de machine learning para predecir las leyes (`pFe`, `pCu`, `pZn`, `pMo`) a partir de las intensidades ortogonales (`*_ortho`).